In [0]:
# Databricks notebook source
#final quality check for your Databricks pipeline
# TASK 4: Production Validation and Reconciliation
# Validates the gold star schema (dim_carrier, dim_warehouse, fact_shipment,
# fact_delivery_attempt, fact_route) against silver, and the gold_* aggregate
# tables written by 08_gold_layer against the facts.

from pyspark.sql import functions as F

SILVER = "logistics.silver"
GOLD = "logistics.gold"

# --------------------------------------------------
# 1. Check required tables
# --------------------------------------------------

required_tables = [
    f"{SILVER}.shipment_orders",
    f"{SILVER}.delivery_attempts",
    f"{SILVER}.silver_shipment_current",
    f"{SILVER}.silver_shipment_status_history",
    f"{SILVER}.route_master",
    f"{GOLD}.dim_carrier",
    f"{GOLD}.dim_warehouse",
    f"{GOLD}.fact_shipment",
    f"{GOLD}.fact_delivery_attempt",
    f"{GOLD}.fact_route",
    f"{GOLD}.gold_shipments_by_status",
    f"{GOLD}.gold_carrier_performance",
    f"{GOLD}.gold_daily_shipments",
    f"{GOLD}.gold_destination_city_summary",
    f"{GOLD}.gold_service_level_summary",
    f"{GOLD}.gold_delivery_attempt_summary",
    f"{GOLD}.gold_city_delivery_success",
    f"{GOLD}.gold_status_transitions",
    f"{GOLD}.gold_cdc_operation_summary"
]

missing = [
    table for table in required_tables
    if not spark.catalog.tableExists(table)
]

if missing:
    raise RuntimeError(
        f"Missing tables. Run the pipeline notebooks first: {missing}"
    )

# --------------------------------------------------
# 2. Load tables
# --------------------------------------------------

orders = spark.table(f"{SILVER}.shipment_orders")
attempts = spark.table(f"{SILVER}.delivery_attempts")
current = spark.table(f"{SILVER}.silver_shipment_current")
history = spark.table(f"{SILVER}.silver_shipment_status_history")
routes = spark.table(f"{SILVER}.route_master")
dim_carrier = spark.table(f"{GOLD}.dim_carrier")
dim_warehouse = spark.table(f"{GOLD}.dim_warehouse")
fact_shipment = spark.table(f"{GOLD}.fact_shipment")
fact_attempt = spark.table(f"{GOLD}.fact_delivery_attempt")
fact_route = spark.table(f"{GOLD}.fact_route")

gold = {
    t.split(".")[-1]: spark.table(t)
    for t in required_tables if t.startswith(f"{GOLD}.gold_")
}

# --------------------------------------------------
# 3. Validation helpers (collect all failures, fail once)
# --------------------------------------------------

failures = []

def check(condition, message):
    if not condition:
        failures.append(message)

def col_sum(df, col):
    return df.agg(F.sum(col).alias("v")).first()["v"] or 0

# --------------------------------------------------
# 4. Schema validation
# --------------------------------------------------

required_columns = {
    "gold_shipments_by_status": {"shipment_status", "shipment_count", "total_shipping_cost", "avg_shipping_cost", "avg_weight_kg"},
    "gold_carrier_performance": {"carrier_id", "carrier_name", "carrier_tier", "service_region", "shipment_count", "total_revenue", "total_weight_kg", "delivered_count", "avg_cost_per_kg"},
    "gold_daily_shipments": {"shipment_date", "shipment_count", "total_revenue", "total_weight_kg"},
    "gold_destination_city_summary": {"destination_city", "shipment_count", "avg_shipping_cost", "avg_weight_kg"},
    "gold_service_level_summary": {"service_level", "shipment_count", "avg_shipping_cost", "avg_weight_kg"},
    "gold_delivery_attempt_summary": {"attempt_result", "failure_reason", "attempt_count", "shipment_count"},
    "gold_city_delivery_success": {"delivery_city", "attempt_count", "successful_attempts", "success_rate_pct"},
    "gold_status_transitions": {"from_status", "to_status", "transition_count"},
    "gold_cdc_operation_summary": {"cdc_operation", "shipment_count", "pct_of_total"},
}

for table, cols in required_columns.items():
    missing_cols = cols - set(gold[table].columns)
    check(not missing_cols, f"{table} is missing columns: {missing_cols}")

# --------------------------------------------------
# 5. Gold completeness and non-empty checks
# --------------------------------------------------

gold_row_counts = {}
for table, df in gold.items():
    gold_row_counts[table] = df.count()
    check(gold_row_counts[table] > 0, f"{table} is empty")
    null_counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).first().asDict()
    bad = {c: n for c, n in null_counts.items() if n}
    check(not bad, f"{table} failed completeness check: {bad}")

# --------------------------------------------------
# 5b. Star schema checks (dims and facts)
# --------------------------------------------------

star_tables = {
    "dim_carrier": (dim_carrier, "carrier_sk"),
    "dim_warehouse": (dim_warehouse, "warehouse_sk"),
    "fact_shipment": (fact_shipment, "shipment_sk"),
    "fact_delivery_attempt": (fact_attempt, "attempt_sk"),
    "fact_route": (fact_route, "route_sk"),
}
star_row_counts = {}
for name, (df, sk) in star_tables.items():
    star_row_counts[name] = df.count()
    check(star_row_counts[name] > 0, f"{name} is empty")
    dup_sk = df.groupBy(sk).count().filter("count > 1").count()
    check(dup_sk == 0, f"{name} has {dup_sk} duplicate {sk} values")
    null_counts = df.select(
        [F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]
    ).first().asDict()
    bad = {c: n for c, n in null_counts.items() if n}
    check(not bad, f"{name} failed completeness check: {bad}")

dup_carriers = dim_carrier.groupBy("carrier_id").count().filter("count > 1").count()
check(dup_carriers == 0, f"dim_carrier has {dup_carriers} duplicate carrier_id values")

# Referential integrity: fact foreign keys must resolve to a dimension row
fk_checks = [
    ("fact_shipment.carrier_sk", fact_shipment, "carrier_sk", dim_carrier, "carrier_sk"),
    ("fact_shipment.origin_warehouse_sk", fact_shipment, "origin_warehouse_sk", dim_warehouse, "warehouse_sk"),
    ("fact_delivery_attempt.carrier_sk", fact_attempt, "carrier_sk", dim_carrier, "carrier_sk"),
    ("fact_route.origin_warehouse_sk", fact_route, "origin_warehouse_sk", dim_warehouse, "warehouse_sk"),
]
for name, fdf, fk, ddf, pk in fk_checks:
    orphans = fdf.join(ddf.select(F.col(pk).alias(fk)), fk, "left_anti").count()
    check(orphans == 0, f"{name}: {orphans} keys not found in dimension")

# Facts reconcile to their silver sources
check(star_row_counts["fact_shipment"] == current.count(),
      f"fact_shipment ({star_row_counts['fact_shipment']}) != silver current ({current.count()})")
check(star_row_counts["fact_delivery_attempt"] == attempts.count(),
      f"fact_delivery_attempt ({star_row_counts['fact_delivery_attempt']}) != silver attempts ({attempts.count()})")
check(star_row_counts["fact_route"] == routes.count(),
      f"fact_route ({star_row_counts['fact_route']}) != silver route_master ({routes.count()})")
missing_fact_ids = current.select("shipment_id").subtract(fact_shipment.select("shipment_id")).count()
check(missing_fact_ids == 0, f"{missing_fact_ids} silver current shipments missing from fact_shipment")

# --------------------------------------------------
# 6. Silver current / history consistency
# --------------------------------------------------

current_count = current.count()
history_count = history.count()

duplicate_current = (
    current.groupBy("shipment_id").count().filter("count > 1").count()
)
check(duplicate_current == 0, f"{duplicate_current} duplicate shipment IDs in current status table")

check(
    current_count == history_count,
    f"Current ({current_count}) and status history ({history_count}) row counts differ"
)

orphan_current = current.select("shipment_id").subtract(orders.select("shipment_id")).count()
check(orphan_current == 0, f"{orphan_current} current shipments not found in silver shipment_orders")

# --------------------------------------------------
# 7. Shipment gold reconciliation (vs fact_shipment / silver_shipment_current)
# --------------------------------------------------

for table in ["gold_shipments_by_status", "gold_daily_shipments",
              "gold_destination_city_summary", "gold_service_level_summary"]:
    total = col_sum(gold[table], "shipment_count")
    check(total == current_count, f"{table}: shipment_count={total}, silver current={current_count}")

silver_cost = float(col_sum(current, "shipping_cost"))
fact_cost = float(col_sum(fact_shipment, "shipping_cost"))
gold_cost = float(col_sum(gold["gold_shipments_by_status"], "total_shipping_cost"))
check(abs(silver_cost - fact_cost) <= 0.01,
      f"Shipping cost does not reconcile: silver={silver_cost:.2f}, fact={fact_cost:.2f}")
check(
    abs(fact_cost - gold_cost) <= 0.01 * max(gold_row_counts["gold_shipments_by_status"], 1),
    f"Shipping cost does not reconcile: fact={fact_cost:.2f}, gold={gold_cost:.2f}"
)

expected_carrier_shipments = fact_shipment.join(dim_carrier.select("carrier_sk"), "carrier_sk", "inner").count()
gold_carrier_shipments = col_sum(gold["gold_carrier_performance"], "shipment_count")
check(
    gold_carrier_shipments == expected_carrier_shipments,
    f"gold_carrier_performance: shipment_count={gold_carrier_shipments}, expected={expected_carrier_shipments}"
)

silver_delivered = fact_shipment.filter(F.col("shipment_status") == "DELIVERED") \
    .join(dim_carrier.select("carrier_sk"), "carrier_sk", "inner").count()
gold_delivered = col_sum(gold["gold_carrier_performance"], "delivered_count")
check(
    gold_delivered == silver_delivered,
    f"gold_carrier_performance: delivered_count={gold_delivered}, silver={silver_delivered}"
)

bad_delivered = gold["gold_carrier_performance"].filter(
    F.col("delivered_count") > F.col("shipment_count")
).count()
check(bad_delivered == 0, "delivered_count exceeds shipment_count for some carriers")

# --------------------------------------------------
# 8. Delivery attempt gold reconciliation
# --------------------------------------------------

attempt_count = fact_attempt.count()

for table in ["gold_delivery_attempt_summary", "gold_city_delivery_success"]:
    total = col_sum(gold[table], "attempt_count")
    check(total == attempt_count, f"{table}: attempt_count={total}, silver attempts={attempt_count}")

silver_success = fact_attempt.filter(F.col("attempt_result") == "DELIVERED").count()
gold_success = col_sum(gold["gold_city_delivery_success"], "successful_attempts")
check(
    gold_success == silver_success,
    f"gold_city_delivery_success: successful_attempts={gold_success}, silver={silver_success}"
)

invalid_rates = gold["gold_city_delivery_success"].filter(
    (F.col("success_rate_pct") < 0) | (F.col("success_rate_pct") > 100)
    | (F.col("successful_attempts") > F.col("attempt_count"))
).count()
check(invalid_rates == 0, "success_rate_pct must be between 0 and 100")

# --------------------------------------------------
# 9. Status history gold reconciliation
# --------------------------------------------------

tracked_count = history.filter(F.col("cdc_operation") != "NO_EVENT").count()
gold_transitions = col_sum(gold["gold_status_transitions"], "transition_count")
check(
    gold_transitions == tracked_count,
    f"gold_status_transitions: transition_count={gold_transitions}, tracked shipments={tracked_count}"
)

cdc = gold["gold_cdc_operation_summary"]
gold_cdc_total = col_sum(cdc, "shipment_count")
check(
    gold_cdc_total == history_count,
    f"gold_cdc_operation_summary: shipment_count={gold_cdc_total}, status history={history_count}"
)

allowed_cdc = ["UPDATE", "NO_CHANGE", "NO_EVENT"]
invalid_cdc = cdc.filter(~F.col("cdc_operation").isin(allowed_cdc)).count()
check(invalid_cdc == 0, f"Unexpected cdc_operation values (allowed: {allowed_cdc})")

pct_total = float(col_sum(cdc, "pct_of_total"))
check(abs(pct_total - 100) <= 0.1, f"gold_cdc_operation_summary pct_of_total sums to {pct_total}")

# --------------------------------------------------
# 10. Final report
# --------------------------------------------------

report = {
    "silver_current_shipments": current_count,
    "silver_status_history_rows": history_count,
    "duplicate_current_status": duplicate_current,
    "carrier_shipments_expected": expected_carrier_shipments,
    "carrier_shipments_gold": gold_carrier_shipments,
    "fact_delivery_attempts": attempt_count,
    "tracked_shipments": tracked_count,
    "gold_transitions": gold_transitions,
    "star_row_counts": star_row_counts,
    "gold_row_counts": gold_row_counts,
}

if failures:
    print("TASK 4 VALIDATION: FAIL")
    print(report)
    raise AssertionError("Validation failed:\n- " + "\n- ".join(failures))

print("TASK 4 VALIDATION: PASS")
print(report)